# 04 — Evaluation: scorecard, percentile-rank monotonicity, regret stability

**Project:** Dynamic Retention Benchmarking (H03)

We close the loop with three deliverables:
1. Per-sector MAE + R² scorecard.
2. Percentile-rank monotonicity test — `/benchmark` should never invert percentile order on a same-sector input.
3. Bandit regret stability across seeds.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from retention_bench.data import generate, write_processed, PROCESSED, ACTIONS
from retention_bench.features import NUMERIC_FEATURES, add_engineered
from retention_bench import models
from retention_bench.models import EpsilonGreedyBandit, predict_retention

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

In [ ]:
PARQUET = PROCESSED / 'retention_panel.parquet'
df = pd.read_parquet(PARQUET) if PARQUET.exists() else generate()
df = add_engineered(df)
try:
    ridge_per_sector = models.load('ridge_per_sector.joblib')
    bandit = models.load('bandit.joblib')
except FileNotFoundError:
    models.train_all()
    ridge_per_sector = models.load('ridge_per_sector.joblib')
    bandit = models.load('bandit.joblib')

## 1. Held-out scorecard

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
perm = rng.permutation(len(df))
df_test = df.iloc[perm[int(0.8*len(df)):]].reset_index(drop=True)
y_pred = np.array([predict_retention(ridge_per_sector, r.to_dict()) for _, r in df_test.iterrows()])
scorecard = pd.Series({
    'MAE': mean_absolute_error(df_test['retention_rate'], y_pred),
    'R^2': r2_score(df_test['retention_rate'], y_pred),
    'mean predicted': y_pred.mean(),
    'mean observed': df_test['retention_rate'].mean(),
}).round(4)
scorecard.to_frame('value')

## 2. Per-sector MAE bar chart

In [ ]:
df_test = df_test.assign(pred=y_pred)
by_sector = (
    df_test.groupby('sector')
    .apply(lambda x: mean_absolute_error(x['retention_rate'], x['pred']))
    .sort_values()
)
fig, ax = plt.subplots(figsize=(8, 3.6))
by_sector.plot(kind='bar', ax=ax, color='#3a7ca5')
ax.set_ylabel('MAE')
ax.set_title('Per-sector held-out MAE')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## 3. Percentile-rank monotonicity

If org A has higher predicted retention than org B in the same sector, the API's percentile-rank for A must also be higher. We sample triplets and check.

In [ ]:
def percentile_rank(sector_df, pred_value):
    return float((sector_df['retention_rate'] < pred_value).mean())

violations = 0
tested = 0
rng_t = np.random.default_rng(0)
for sector in df_test['sector'].unique():
    sector_train = df[df['sector'] == sector]
    sub = df_test[df_test['sector'] == sector].sort_values('pred')
    if len(sub) < 4:
        continue
    a, b = sub.iloc[0], sub.iloc[-1]
    pa = percentile_rank(sector_train, a['pred'])
    pb = percentile_rank(sector_train, b['pred'])
    tested += 1
    if not (pa <= pb):
        violations += 1
print(f'{violations}/{tested} sectors violated percentile-rank monotonicity')

## 4. Bandit regret stability across seeds

We run the bandit with three different exploration seeds and compare the cumulative-regret curves. Stable regret across seeds is a soft sign that the action ranking is reproducible.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
best_arm = (
    pd.DataFrame([
        {'sector': s, 'best_mean': max(bandit.means.get((s, a), 0.0) for a in ACTIONS)}
        for s in df['sector'].unique()
    ]).set_index('sector')['best_mean'].to_dict()
)
for seed in [0, 1, 2]:
    rng_b = np.random.default_rng(seed)
    regret = []
    for _, row in df_test.iterrows():
        chosen = bandit.choose(row['sector'], rng=rng_b)
        chosen_mean = bandit.means.get((row['sector'], chosen), 0.0)
        regret.append(max(best_arm.get(row['sector'], 0.0) - chosen_mean, 0.0))
    ax.plot(np.cumsum(regret), label=f'seed={seed}')
ax.set_title('Cumulative regret across seeds')
ax.set_xlabel('row index'); ax.set_ylabel('cum. regret')
ax.legend()
plt.tight_layout(); plt.show()

## 5. Per-sector top-3 action recommendations

The dashboard surfaces a *ranked* top-3 — the HRBP needs alternatives, not a single answer. We render the top-3 per sector for the model card.

In [ ]:
rows = []
for s in df['sector'].unique():
    for r in bandit.rank(s)[:3]:
        rows.append({'sector': s, **r})
top3 = pd.DataFrame(rows)
top3.head(12)

## 6. Calibration of predicted retention vs observed

In [ ]:
bins = pd.qcut(df_test['pred'], q=10, duplicates='drop')
rel = df_test.groupby(bins).agg(
    pred=('pred', 'mean'), obs=('retention_rate', 'mean'), n=('retention_rate', 'size')
).reset_index(drop=True).round(3)
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.plot(rel['pred'], rel['obs'], marker='o')
ax.plot([rel['pred'].min(), rel['pred'].max()], [rel['pred'].min(), rel['pred'].max()], color='grey', ls=':')
ax.set_xlabel('mean predicted retention'); ax.set_ylabel('mean observed')
ax.set_title('Per-decile reliability')
plt.tight_layout(); plt.show()
rel

## 7. Release-readiness checklist

| Gate | Target | Result |
|---|---|---|
| Held-out MAE | ≤ 0.04 | see scorecard |
| R² | ≥ 0.45 | see scorecard |
| Percentile-rank monotonicity | 0 violations | property test |
| Bandit regret stable across seeds | yes | curves above |

Documented next step (`docs/04_evaluation.md`): replace ε-greedy with LinUCB or Thompson sampling once we have enough live HR-action telemetry to estimate the per-arm posterior reliably.